Vamos ahora a tratar las imputaciones registradas con el sistema MES

En las imputaciones es donde esta el detalle real de ejecución (qué máquina, qué tiempos reales, quién y cuándo).

In [1]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [2]:
ruta_imputations = os.path.join("res\\raw", "imputaciones.csv")
print(ruta_imputations)

res\raw\imputaciones.csv


In [8]:
columns_imputations = ["FechaCreacionDte",	"smNodoId",	"smNodoNme",	"smOrdenId",	"smMaterialId",	"smOrdenNme",	"smFaseId",	"smFaseNme",	"CantidadManualOKNbr",	"CantidadManualNOKNbr",	"MinutosRealNbr",	"MinutosPreparacionNbr",	"MinutosProduccionNbr",	"PorcentajeDedicacionNbr",	"ProductivoInd",	"smIncidenciaId",	"smIncidenciaNme",	"smFasePersonalizado1Txt",	"smNodoId2",	"smNodoPersonalizado1Nbr",	"smNodoPersonalizado2Nbr",	"ssEnumFaseEstadoId",	"SmFaseUid",	"TraspasadoInd",	"JsonBodyTxt",	"JsonResponseTxt"]

pd.set_option("display.max_columns", None)  
pd.set_option("display.width", None)        

df_imputations = pd.read_csv(ruta_imputations, sep=";", header=None, names=columns_imputations)

C:\Users\jaume\AppData\Local\Temp\ipykernel_19516\1223014052.py:6: DtypeWarning: Columns (0: smFaseId, 1: smIncidenciaId, 2: smIncidenciaNme) have mixed types. Specify dtype option on import or set low_memory=False.
  df_imputations = pd.read_csv(ruta_imputations, sep=";", header=None, names=columns_imputations)


In [9]:
display(df_imputations.head(1))

,FechaCreacionDte,smNodoId,smNodoNme,smOrdenId,smMaterialId,smOrdenNme,smFaseId,smFaseNme,CantidadManualOKNbr,CantidadManualNOKNbr,MinutosRealNbr,MinutosPreparacionNbr,MinutosProduccionNbr,PorcentajeDedicacionNbr,ProductivoInd,smIncidenciaId,smIncidenciaNme,smFasePersonalizado1Txt,smNodoId2,smNodoPersonalizado1Nbr,smNodoPersonalizado2Nbr,ssEnumFaseEstadoId,SmFaseUid,TraspasadoInd,JsonBodyTxt,JsonResponseTxt
0,2026-08-05 00:57:09.823,TN09,TPP DOOSAN PUMA 240 (TPP) Torn C.N.C,OF2606156,PE1JNCX0012,Portacápsulas F704501049704(150713),30,TPP,0.0,0.0,0.0,0.0,0.0,0.0,1,NaN,NaN,NaN,TN09,NaN,NaN,INICIADA_OLANET,C0DF4C4C-2E19-483E-B9CD-6506FC7BABA0,1,"{""CodCompany"":""100"",""CodSite"":""01"",""Imputation...","{""Result"":true,""errorDetails"":[],""newIDImputat..."


In [10]:
display(df_imputations.shape)

(196576, 26)

## Limpieza de atributos
Eliminar del dataset aquellos atributos prescindibles. 


In [ ]:
def eliminar_columnas(df):
    """
    Elimina de df_imputations las columnas marcadas como innecesarias.
    Devuelve un DataFrame nuevo (no modifica el original).
    """

    cols_sobran = [
        "JsonResponseTxt", "JsonBodyTxt", "TraspasadoInd", "ssEnumFaseEstadoId", 
        "smNodoPersonalizado2Nbr", "smNodoPersonalizado1Nbr", "smNodoId2",
        "smFasePersonalizado1Txt", "PorcentajeDedicacionNbr", "CantidadManualOKNbr",
        "CantidadManualNOKNbr", "smFaseId",
        
    ]

    cols_a_quitar = [c for c in cols_sobran if c in columns_imputations]
    df_out = df.drop(columns=cols_a_quitar)

    # Aviso por si alguna de la lista no se encontró (typo o nombre distinto)
    no_encontradas = [c for c in cols_sobran if c not in df_imputations.columns]
    if no_encontradas:
        print(f"Aviso: {len(no_encontradas)} columnas de la lista no estaban en df: {no_encontradas}")

    print(f"Eliminadas {len(cols_a_quitar)} columnas. "
          f"Quedan {df_out.shape[1]} columnas, {df_out.shape[0]} filas.")
    return df_out

In [47]:
df_imputations_limpieza = eliminar_columnas(df_imputations)

Eliminadas 11 columnas. Quedan 15 columnas, 196576 filas.


In [48]:
display(df_imputations_limpieza.head(1))

,FechaCreacionDte,smNodoId,smNodoNme,smOrdenId,smMaterialId,smOrdenNme,smFaseId,smFaseNme,MinutosRealNbr,MinutosPreparacionNbr,MinutosProduccionNbr,ProductivoInd,smIncidenciaId,smIncidenciaNme,SmFaseUid
0,2026-08-05 00:57:09.823,TN09,TPP DOOSAN PUMA 240 (TPP) Torn C.N.C,OF2606156,PE1JNCX0012,Portacápsulas F704501049704(150713),30,TPP,0.0,0.0,0.0,1,NaN,NaN,C0DF4C4C-2E19-483E-B9CD-6506FC7BABA0


### Limpieza por intervalo de fechas
Trabajamos dentro del rango entre junio 2025 y antes de agosto 2026.

In [49]:
def filtrar_desde_fecha(df, fecha_desde, columna='FechaCreacionDte'):
    """Conserva solo las instancaias cuya fecha en 'columna' sea >= fecha_desde.
    Convierte la columna a datetime por seguridad (sirve si viene como texto).
    Devuelve un DataFrame nuevo."""

    n_inicial = df.shape[0]
    df_out = df.copy()

    # Convertir a datetime de forma robusta. errors='coerce' -> fechas no
    # parseables se vuelven NaT (nulo temporal) en vez de reventar.
    df_out[columna] = pd.to_datetime(df_out[columna], errors='coerce')

    # Avisar si alguna fecha no se pudo interpretar
    n_nat = df_out[columna].isna().sum()
    if n_nat:
        print(f"Aviso: {n_nat} filas con fecha no interpretable (NaT). "
              f"Quedan fuera del filtro >= {fecha_desde}.")

    limite = pd.to_datetime(fecha_desde)
    df_out = df_out[df_out[columna] >= limite].copy()

    print(f"Partida: {n_inicial} fases")
    print(f"Conservadas (desde {fecha_desde}): {df_out.shape[0]} "
          f"(descartadas {n_inicial - df_out.shape[0]})")
    return df_out

In [50]:
df_imputations_limpieza = filtrar_desde_fecha(df_imputations_limpieza, "2025-06-01")

Partida: 196576 fases
Conservadas (desde 2025-06-01): 141329 (descartadas 55247)


In [20]:
def filtrar_hasta_fecha(df, fecha_hasta, columna='FechaCreacionDte'):
    """Conserva solo las instancaias cuya fecha en 'columna' sea =< fecha_hasta.
    Convierte la columna a datetime por seguridad (sirve si viene como texto).
    Devuelve un DataFrame nuevo."""

    n_inicial = df.shape[0]
    df_out = df.copy()

    # Convertir a datetime de forma robusta. errors='coerce' -> fechas no
    # parseables se vuelven NaT (nulo temporal) en vez de reventar.
    df_out[columna] = pd.to_datetime(df_out[columna], errors='coerce')

    # Avisar si alguna fecha no se pudo interpretar
    n_nat = df_out[columna].isna().sum()
    if n_nat:
        print(f"Aviso: {n_nat} filas con fecha no interpretable (NaT). "
              f"Quedan fuera del filtro =< {fecha_hasta}.")

    limite = pd.to_datetime(fecha_hasta)
    df_out = df_out[df_out[columna] <= limite].copy()

    print(f"Partida: {n_inicial} fases")
    print(f"Conservadas (hasta {fecha_hasta}): {df_out.shape[0]} "
          f"(descartadas {n_inicial - df_out.shape[0]})")
    return df_out

In [51]:
df_imputations_limpieza = filtrar_hasta_fecha(df_imputations_limpieza, "2026-08-01")

Partida: 141329 fases
Conservadas (hasta 2026-08-01): 140485 (descartadas 844)


Voy a guarda la tabla de las imputaciones en `res`

In [53]:
import pyarrow as pa

df_imputations_limpieza["smFaseId"] = df_imputations_limpieza["smFaseId"].astype(str)

df_imputations_limpieza.to_parquet("res/procesed/df_imputations_limpieza.parquet")

Verificar que todas las imputaciones de una misma fase se hayan hecho en la misma máquina.   
Para cada imputación podemos saber en que nodo se ha realizado, un nodo puede ser una maquina o un centro de trabajo.

In [54]:
maquinas_por_fase = df_imputations_limpieza.groupby("SmFaseUid")["smNodoId"].nunique()
conteos = maquinas_por_fase.value_counts()
print(conteos)

smNodoId
1    45176
2     3398
3      180
4       12
5        4
6        1
Name: count, dtype: int64
